# 4 · Model Comparison and Benchmarking

All models evaluated on the **same** repaired validation split. The test split is reserved for the final model alone.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # import the project's src/ package
import pandas as pd
from IPython.display import Image, display
from src import paths


In [ ]:
pd.read_csv(paths.TRAINING_OUTPUT_DIR / 'experiment_log.csv')[
    ['experiment_id','model','epochs_run','best_epoch','batch','map50','map50_95','recall','duration']]

## Benchmark and the recall-weighted selection

Selection is **not** raw mAP: `0.35·mAP50-95 + 0.25·recall + 0.25·smoke_recall + 0.15·speed`. Missing a fire costs more than a false alarm, and smoke is both the harder class and the earlier warning.

In [ ]:
pd.read_csv(paths.BENCHMARK_OUTPUT_DIR / 'benchmark_table.csv')

In [ ]:
display(Image(str(paths.BENCHMARK_OUTPUT_DIR / 'benchmark_chart.png')))
json.loads((paths.BENCHMARK_OUTPUT_DIR / 'selection_report.json').read_text())

## Fair equal-epoch comparison

The models were trained for different numbers of epochs (VRAM and time limits), so their final numbers also encode training length. Because validation metrics are logged every epoch, we can compare them at the same epoch.

In [ ]:
from src.train import metrics_at_epoch
log = pd.read_csv(paths.TRAINING_OUTPUT_DIR / 'experiment_log.csv').set_index('experiment_id')
for exp in [e for e in ['e2_stronger_v8s','e3_compare_11n'] if e in log.index]:
    n = int(log.loc[exp,'epochs_run'])
    base = metrics_at_epoch(paths.TRAINING_OUTPUT_DIR/'e1_baseline_v8n'/'results.csv', n)
    print(f"at epoch {n}:  YOLOv8n mAP50={base['map50']:.3f}  |  "
          f"{exp} mAP50={log.loc[exp,'map50']:.3f}")